# 迁移学习：校园建筑用能负荷预测

本notebook实现三种迁移学习策略，将源域建筑训练的LSTM模型迁移到目标域建筑。

## 迁移策略
1. **预训练-微调（Full Fine-tuning）**：在源域数据上预训练模型，然后在目标域上微调所有参数
2. **冻结部分层微调（Partial Fine-tuning）**：冻结模型前几层（LSTM特征提取层），仅微调靠近输出的层
3. **特征提取+新分类器（Feature Extractor + New Regressor）**：将预训练模型作为固定特征提取器，训练新的回归层

## 流程
```
data_preprocess.ipynb → similarity_analysis.ipynb → prepare_transfer_data.ipynb → transfer_learning.ipynb
```

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
import joblib

# 从 src 模块导入共享配置和函数
from src import (
    # preprocessing
    load_data,
    create_sequences,
    # models
    LSTMPredictor,
    FeatureExtractorRegressor,
    # training
    set_seed,
    evaluate,
    LoadDataset,
    EarlyStopping,
    run_epoch,
    predict,
    # visualization
    plot_training_history,
    plot_predictions,
    # config
    BASE_DIR,
    MODEL_DIR,
    SCALER_DIR,
    TARGET_COL,
    TIME_COL,
    SEED,
)

# ===== 本 notebook 的超参数配置 =====
LOOKBACK = 24   # 输入窗口长度（小时）
HORIZON = 1     # 预测步长（小时）
BATCH_SIZE = 72
EPOCHS = 50
HIDDEN_SIZE = 128

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.style.use('default')

print(f"Using device: {device}")
print(f"配置: LOOKBACK={LOOKBACK}, HORIZON={HORIZON}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, HIDDEN_SIZE={HIDDEN_SIZE}")

## 数据准备

In [ ]:
# 确保目录存在
MODEL_DIR.mkdir(exist_ok=True)

# 数据路径
SOURCE_TRAIN_PATH = BASE_DIR / 'source_train_std.csv'  # 源域全量数据
TARGET_TRAIN_PATH = BASE_DIR / 'target_train_std.csv'
TARGET_VAL_PATH = BASE_DIR / 'target_val_std.csv'
TARGET_TEST_PATH = BASE_DIR / 'target_test_std.csv'

print("数据路径配置完成")
print(f"  源域（全量预训练）: {SOURCE_TRAIN_PATH.name}")
print(f"  目标域训练集: {TARGET_TRAIN_PATH.name}")
print(f"  目标域验证集: {TARGET_VAL_PATH.name}")
print(f"  目标域测试集: {TARGET_TEST_PATH.name}")

In [ ]:
set_seed(SEED)

# 读取源域数据（全量数据）
source_full_df = load_data(SOURCE_TRAIN_PATH).dropna(subset=[TARGET_COL]).reset_index(drop=True)
print(f"\n源域全量数据: {source_full_df.shape}")

# 从源域全量数据中划分 10% 作为验证集（用于早停）
source_val_ratio = 0.10
source_n = len(source_full_df)
source_val_end = int(source_n * source_val_ratio)

source_train_df = source_full_df.iloc[:-source_val_end].reset_index(drop=True)
source_val_df = source_full_df.iloc[-source_val_end:].reset_index(drop=True)

print(f"  训练集: {source_train_df.shape}")
print(f"  验证集: {source_val_df.shape} (从全量数据末尾划分 {source_val_ratio*100:.0f}%)")

# 读取目标域数据
target_train_df = load_data(TARGET_TRAIN_PATH).dropna(subset=[TARGET_COL]).reset_index(drop=True)
target_val_df = load_data(TARGET_VAL_PATH).dropna(subset=[TARGET_COL]).reset_index(drop=True)
target_test_df = load_data(TARGET_TEST_PATH).dropna(subset=[TARGET_COL]).reset_index(drop=True)

print(f"\n目标域数据形状:")
print(f"  训练集: {target_train_df.shape}")
print(f"  验证集: {target_val_df.shape}")
print(f"  测试集: {target_test_df.shape}")

In [ ]:
# 确定特征列（除时间戳外）
FEATURE_COLS = [c for c in source_train_df.columns if c != TIME_COL]
input_size = len(FEATURE_COLS)

print(f"特征维度: {input_size}")

In [ ]:
# 构造序列
print("构造序列...")
X_source_train, y_source_train = create_sequences(source_train_df, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON, TIME_COL)
X_source_val, y_source_val = create_sequences(source_val_df, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON, TIME_COL)

X_target_train, y_target_train = create_sequences(target_train_df, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON, TIME_COL)
X_target_val, y_target_val = create_sequences(target_val_df, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON, TIME_COL)
X_target_test, y_target_test = create_sequences(target_test_df, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON, TIME_COL)

print(f"源域训练序列: {X_source_train.shape}")
print(f"源域验证序列: {X_source_val.shape}")
print(f"目标域训练序列: {X_target_train.shape}")
print(f"目标域验证序列: {X_target_val.shape}")
print(f"目标域测试序列: {X_target_test.shape}")

In [ ]:
# 创建数据加载器
source_train_dataset = LoadDataset(X_source_train, y_source_train)
source_val_dataset = LoadDataset(X_source_val, y_source_val)
source_train_loader = DataLoader(source_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
source_val_loader = DataLoader(source_val_dataset, batch_size=BATCH_SIZE, shuffle=False)

target_train_dataset = LoadDataset(X_target_train, y_target_train)
target_val_dataset = LoadDataset(X_target_val, y_target_val)
target_test_dataset = LoadDataset(X_target_test, y_target_test)
target_train_loader = DataLoader(target_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
target_val_loader = DataLoader(target_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
target_test_loader = DataLoader(target_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("数据加载器创建完成")

# 加载目标域的scaler
target_scaler = joblib.load(SCALER_DIR / 'target_load_scaler.joblib')
print("已加载目标域scaler")

# 定义损失函数
criterion = nn.L1Loss()  # MAE loss

## 第一阶段：在源域上训练基础模型

In [ ]:
print("="*60)
print("第一阶段：在源域上训练基础模型")
print("="*60)

# 初始化模型
base_model = LSTMPredictor(input_size=input_size, hidden_size=HIDDEN_SIZE).to(device)

# 优化器和调度器
optimizer = torch.optim.Adam(base_model.parameters(), weight_decay=1e-3)
lr_scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=4, min_lr=1e-5)
early_stopping = EarlyStopping(patience=5, restore_best_weights=True)

train_losses = []
val_losses = []

print("\n开始在源域上训练...")
for epoch in range(EPOCHS):
    train_loss = run_epoch(base_model, source_train_loader, criterion, device, optimizer)
    val_loss = run_epoch(base_model, source_val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    lr_scheduler.step(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:2d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    
    if early_stopping(val_loss, base_model):
        print(f"Early stopping at epoch {epoch+1}")
        break

early_stopping.restore(base_model)
print(f"\n源域训练完成，共 {len(train_losses)} 个 epoch")

# 保存预训练模型
torch.save(base_model.state_dict(), MODEL_DIR / 'pretrained_source.pt')
print(f"预训练模型已保存: {MODEL_DIR / 'pretrained_source.pt'}")

# 可视化训练过程
plot_training_history(train_losses, val_losses, 'Source Domain Training History')

## 第二阶段：迁移学习实验

### 基线模型：从零训练（Baseline）

作为对照组，不使用任何预训练权重，从零开始在目标域数据上训练模型。

In [ ]:
print("="*60)
print("基线模型：从零训练（Baseline）")
print("="*60)

baseline_model = LSTMPredictor(input_size=input_size, hidden_size=HIDDEN_SIZE).to(device)

optimizer_baseline = torch.optim.Adam(baseline_model.parameters(), weight_decay=1e-3)
early_stopping_baseline = EarlyStopping(patience=5, restore_best_weights=True)
lr_scheduler_baseline = ReduceLROnPlateau(optimizer_baseline, factor=0.5, patience=4, min_lr=1e-5)

train_losses_baseline = []
val_losses_baseline = []

print("\n开始在目标域上从零训练...")
for epoch in range(EPOCHS):
    train_loss = run_epoch(baseline_model, target_train_loader, criterion, device, optimizer_baseline)
    val_loss = run_epoch(baseline_model, target_val_loader, criterion, device)
    
    train_losses_baseline.append(train_loss)
    val_losses_baseline.append(val_loss)
    
    lr_scheduler_baseline.step(val_loss)
    
    if early_stopping_baseline(val_loss, baseline_model):
        break

early_stopping_baseline.restore(baseline_model)
print(f"Baseline训练完成，共 {len(train_losses_baseline)} 个 epoch")

# 在目标域测试集上评估
y_pred_baseline = predict(baseline_model, target_test_loader, device)
metrics_baseline = evaluate(y_target_test, y_pred_baseline, target_scaler)

print("\n基线模型测试集性能：")
print(f"MAE: {metrics_baseline['MAE']:.4f}")
print(f"RMSE: {metrics_baseline['RMSE']:.4f}")
print(f"MAPE: {metrics_baseline['MAPE']:.4f}%")
print(f"R2: {metrics_baseline['R2']:.4f}")

plot_training_history(train_losses_baseline, val_losses_baseline, 'Baseline Model Training History')
plot_predictions(y_target_test, y_pred_baseline, 'Baseline Model Predictions', target_scaler)

### 策略1：预训练-微调（Full Fine-tuning）

加载源域预训练模型，在目标域训练集上微调所有参数。

In [ ]:
print("="*60)
print("策略1: 预训练-微调（Full Fine-tuning）")
print("="*60)

# 加载预训练模型
finetuned_full = LSTMPredictor(input_size=input_size, hidden_size=HIDDEN_SIZE).to(device)
finetuned_full.load_state_dict(torch.load(MODEL_DIR / 'pretrained_source.pt', map_location=device))
print(f"已加载预训练模型: {MODEL_DIR / 'pretrained_source.pt'}")

# 微调所有参数
optimizer_ft_full = torch.optim.Adam(finetuned_full.parameters(), lr=1e-4, weight_decay=1e-3)
early_stopping_ft_full = EarlyStopping(patience=5, restore_best_weights=True)
lr_scheduler_ft_full = ReduceLROnPlateau(optimizer_ft_full, factor=0.5, patience=4, min_lr=1e-6)

train_losses_ft_full = []
val_losses_ft_full = []

print("\n开始目标域微调（全参数）...")
for epoch in range(EPOCHS):
    train_loss = run_epoch(finetuned_full, target_train_loader, criterion, device, optimizer_ft_full)
    val_loss = run_epoch(finetuned_full, target_val_loader, criterion, device)
    
    train_losses_ft_full.append(train_loss)
    val_losses_ft_full.append(val_loss)
    
    lr_scheduler_ft_full.step(val_loss)
    
    if early_stopping_ft_full(val_loss, finetuned_full):
        break

early_stopping_ft_full.restore(finetuned_full)
print(f"全参数微调完成，共 {len(train_losses_ft_full)} 个 epoch")

# 在目标域测试集上评估
y_pred_ft_full = predict(finetuned_full, target_test_loader, device)
metrics_ft_full = evaluate(y_target_test, y_pred_ft_full, target_scaler)

print("\n全参数微调模型测试集性能：")
print(f"MAE: {metrics_ft_full['MAE']:.4f}")
print(f"RMSE: {metrics_ft_full['RMSE']:.4f}")
print(f"MAPE: {metrics_ft_full['MAPE']:.4f}%")
print(f"R2: {metrics_ft_full['R2']:.4f}")

plot_training_history(train_losses_ft_full, val_losses_ft_full, 'Full Fine-tuning Training History')
plot_predictions(y_target_test, y_pred_ft_full, 'Full Fine-tuning Predictions', target_scaler)

### 策略2：冻结部分层微调（Partial Fine-tuning）

冻结LSTM层（特征提取层），仅微调全连接层。

In [ ]:
print("="*60)
print("策略2: 冻结部分层微调（Partial Fine-tuning）")
print("="*60)

# 加载预训练模型
finetuned_partial = LSTMPredictor(input_size=input_size, hidden_size=HIDDEN_SIZE).to(device)
finetuned_partial.load_state_dict(torch.load(MODEL_DIR / 'pretrained_source.pt', map_location=device))
print(f"已加载预训练模型: {MODEL_DIR / 'pretrained_source.pt'}")

# 冻结LSTM层，只微调fc层
print("\n冻结LSTM层，仅微调全连接层：")
for name, param in finetuned_partial.named_parameters():
    if 'lstm' in name:
        param.requires_grad = False
        print(f"  冻结: {name}")
    else:
        print(f"  可训练: {name}")

# 微调（只微调fc层）
optimizer_ft_partial = torch.optim.Adam(
    filter(lambda p: p.requires_grad, finetuned_partial.parameters()), 
    lr=1e-3, weight_decay=1e-3
)
early_stopping_ft_partial = EarlyStopping(patience=8, restore_best_weights=True)
lr_scheduler_ft_partial = ReduceLROnPlateau(optimizer_ft_partial, factor=0.5, patience=4, min_lr=1e-6)

train_losses_ft_partial = []
val_losses_ft_partial = []

print("\n开始目标域微调（冻结LSTM层）...")
for epoch in range(EPOCHS):
    train_loss = run_epoch(finetuned_partial, target_train_loader, criterion, device, optimizer_ft_partial)
    val_loss = run_epoch(finetuned_partial, target_val_loader, criterion, device)
    
    train_losses_ft_partial.append(train_loss)
    val_losses_ft_partial.append(val_loss)
    
    lr_scheduler_ft_partial.step(val_loss)
    
    if early_stopping_ft_partial(val_loss, finetuned_partial):
        break

early_stopping_ft_partial.restore(finetuned_partial)
print(f"冻结部分层微调完成，共 {len(train_losses_ft_partial)} 个 epoch")

# 在目标域测试集上评估
y_pred_ft_partial = predict(finetuned_partial, target_test_loader, device)
metrics_ft_partial = evaluate(y_target_test, y_pred_ft_partial, target_scaler)

print("\n冻结部分层微调模型测试集性能：")
print(f"MAE: {metrics_ft_partial['MAE']:.4f}")
print(f"RMSE: {metrics_ft_partial['RMSE']:.4f}")
print(f"MAPE: {metrics_ft_partial['MAPE']:.4f}%")
print(f"R2: {metrics_ft_partial['R2']:.4f}")

plot_training_history(train_losses_ft_partial, val_losses_ft_partial, 'Partial Fine-tuning Training History')
plot_predictions(y_target_test, y_pred_ft_partial, 'Partial Fine-tuning Predictions', target_scaler)

### 策略3：特征提取+新分类器（Feature Extractor + New Regressor）

将预训练模型作为固定特征提取器，训练新的回归层。

In [ ]:
print("="*60)
print("策略3: 特征提取+新分类器（Feature Extractor + New Regressor）")
print("="*60)

# 加载预训练模型
pretrained = LSTMPredictor(input_size=input_size, hidden_size=HIDDEN_SIZE).to(device)
pretrained.load_state_dict(torch.load(MODEL_DIR / 'pretrained_source.pt', map_location=device))
print(f"已加载预训练模型: {MODEL_DIR / 'pretrained_source.pt'}")

# 创建特征提取器模型
feature_extractor_model = FeatureExtractorRegressor(pretrained).to(device)
print("已创建特征提取器模型（LSTM层冻结，仅训练新的回归层）")

# 只训练新添加的层
optimizer_fe = torch.optim.Adam(
    filter(lambda p: p.requires_grad, feature_extractor_model.parameters()), 
    lr=1e-3, weight_decay=1e-3
)
early_stopping_fe = EarlyStopping(patience=8, restore_best_weights=True)
lr_scheduler_fe = ReduceLROnPlateau(optimizer_fe, factor=0.5, patience=4, min_lr=1e-6)

train_losses_fe = []
val_losses_fe = []

print("\n开始训练特征提取+新分类器...")
for epoch in range(EPOCHS):
    train_loss = run_epoch(feature_extractor_model, target_train_loader, criterion, device, optimizer_fe)
    val_loss = run_epoch(feature_extractor_model, target_val_loader, criterion, device)
    
    train_losses_fe.append(train_loss)
    val_losses_fe.append(val_loss)
    
    lr_scheduler_fe.step(val_loss)
    
    if early_stopping_fe(val_loss, feature_extractor_model):
        break

early_stopping_fe.restore(feature_extractor_model)
print(f"特征提取+新分类器训练完成，共 {len(train_losses_fe)} 个 epoch")

# 在目标域测试集上评估
y_pred_fe = predict(feature_extractor_model, target_test_loader, device)
metrics_fe = evaluate(y_target_test, y_pred_fe, target_scaler)

print("\n特征提取+新分类器模型测试集性能：")
print(f"MAE: {metrics_fe['MAE']:.4f}")
print(f"RMSE: {metrics_fe['RMSE']:.4f}")
print(f"MAPE: {metrics_fe['MAPE']:.4f}%")
print(f"R2: {metrics_fe['R2']:.4f}")

plot_training_history(train_losses_fe, val_losses_fe, 'Feature Extractor + New Regressor Training History')
plot_predictions(y_target_test, y_pred_fe, 'Feature Extractor + New Regressor Predictions', target_scaler)

## 实验结果汇总

In [ ]:
# 汇总所有方法的性能
summary_df = pd.DataFrame([
    {'策略': 'Baseline (从零训练)', **metrics_baseline},
    {'策略': 'Full Fine-tuning (全参数微调)', **metrics_ft_full},
    {'策略': 'Partial Fine-tuning (冻结LSTM)', **metrics_ft_partial},
    {'策略': 'Feature Extractor (特征提取)', **metrics_fe},
])

print("\n" + "="*60)
print("迁移学习策略性能对比")
print("="*60)
print(summary_df.to_string(index=False))

# 保存结果
summary_df.to_csv(BASE_DIR / 'transfer_learning_results.csv', index=False, encoding='utf-8-sig')
print(f"\n结果已保存到 {BASE_DIR / 'transfer_learning_results.csv'}")

## 保存模型

In [ ]:
# 保存各策略的模型
torch.save(baseline_model.state_dict(), MODEL_DIR / 'transfer_baseline.pt')
torch.save(finetuned_full.state_dict(), MODEL_DIR / 'transfer_full_finetune.pt')
torch.save(finetuned_partial.state_dict(), MODEL_DIR / 'transfer_partial_finetune.pt')
torch.save(feature_extractor_model.state_dict(), MODEL_DIR / 'transfer_feature_extractor.pt')

print(f"所有模型已保存到 {MODEL_DIR}/ 目录")

## 结果分析

根据实验结果分析各迁移学习策略的效果：

1. **Baseline (从零训练)**：作为对照组，展示在目标域数据有限情况下的模型性能

2. **Full Fine-tuning (全参数微调)**：利用源域预训练权重作为初始化，在目标域上微调所有参数。适合目标域数据量适中的情况。

3. **Partial Fine-tuning (冻结LSTM)**：冻结特征提取层，防止过拟合。适合目标域数据量较小的情况。

4. **Feature Extractor (特征提取)**：将预训练模型作为固定特征提取器，仅训练新的回归层。适合目标域数据量极小的情况。